# NB2 — Structural Econometric Estimation

Structural econometrics pipeline, Phase 2. Fits the OLS RV^{1/3} specification on CPI surprise with six controls, bootstraps standard errors, and serializes point / full estimate JSON + pickle for NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 2 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup — spec-hash pre-flight, panel-fingerprint verification, Gate Verdict anchor

This section is NB2's pre-flight. Before any estimator runs, §1 verifies two invariants: (a) the econ-notebook design spec on disk matches the Rev 4 we pre-registered against, and (b) the cleaned weekly panel we are about to estimate on is the byte-identical object NB1 serialized to `env.FINGERPRINT_PATH`. Either check failing halts the notebook — the estimator never runs on silently drifted inputs.

### §1.1 Spec hash pre-flight

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 replication-protocol discipline; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], anti-fishing pre-commitment discipline.

**Why used.** Ankel-Peters et al. §3 require a pre-registered empirical pipeline to emit a machine-verifiable handoff artifact that captures every locked decision before estimation runs. Spec Rev 4 embeds the same logic at document level: the estimation notebook must hash the spec file on execute and assert equality with the hex string we committed against. Simonsohn et al. formalise the complementary anti-fishing guarantee — a specification locked in a hash cannot be silently amended after results are seen.

**Relevance to our results.** Every coefficient, test statistic, and residual diagnostic in §3-§7 below is one of the specifications the spec Rev 4 pre-registered. If the spec file drifts between NB1 authoring and NB2 estimation, the pre-registration guarantee collapses. Embedding the current sha256 as a literal constant — recomputed against the live file in the next code cell — turns that drift into an execution-time failure rather than a silent post-hoc rewrite.

**Connection to simulator.** Layer 2 simulator consumers replay this notebook end-to-end from a pinned commit to re-materialise fitted parameters. The spec-hash assertion is the simulator's guarantee that the estimation surface it consumed matches the spec text it documents against. Any mismatch here invalidates every β̂ the simulator reads downstream.


In [ ]:
# Bootstrap: make env.py, scripts/cleaning.py, and
# scripts/panel_fingerprint.py importable from this notebook's §1
# onward. Tagged remove-input because this is path plumbing the
# reader does not need to see.
#
# Strategy mirrors NB1: locate the Colombia/ directory robustly,
# add it and contracts/scripts/ to sys.path, then import directly.
import sys
from pathlib import Path


def _locate_colombia_dir() -> Path:
    """Find the Colombia/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "notebooks" / "fx_vol_cpi_surprise" / "Colombia"
        if (candidate / "env.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate Colombia/env.py starting from cwd={cwd}"
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / "scripts"

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR):
    _p_str = str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import env  # noqa: E402  — path plumbing must precede these imports


In [ ]:
# §1.1 Spec-hash pre-flight (Decision #0: pre-registered spec lock).
#
# Recomputes sha256 of docs/superpowers/specs/2026-04-17-econ-notebook-design.md
# against the embedded literal _SPEC_SHA256_REV4. Any edit to the spec
# file between NB1 authoring and NB2 execution flips the hash and halts
# the notebook — the estimator never runs on a silently drifted spec.
import hashlib

_SPEC_MD_PATH = (
    _CONTRACTS_DIR
    / "docs"
    / "superpowers"
    / "specs"
    / "2026-04-17-econ-notebook-design.md"
)

# Embedded at authoring time (Task 16). Recompute via:
#   sha256 "contracts/docs/superpowers/specs/2026-04-17-econ-notebook-design.md"
_SPEC_SHA256_REV4 = (
    "5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6"
)

_current_spec_sha256 = hashlib.sha256(_SPEC_MD_PATH.read_bytes()).hexdigest()
assert _current_spec_sha256 == _SPEC_SHA256_REV4, (
    f"Spec Rev 4 drift detected: file {_SPEC_MD_PATH.name} hash "
    f"{_current_spec_sha256} does not match embedded "
    f"{_SPEC_SHA256_REV4}. Halt — do not estimate on a drifted spec."
)

print(f"Spec Rev 4 sha256 verified: {_current_spec_sha256}")


**Interpretation — §1.1.** The spec file sha256 matches the embedded Rev 4 lock exactly. This means every methodological claim in §2-§7 below — OLS on `rv_cuberoot` against `cpi_surprise_ar1` with the six-control nested ladder, Newey-West HAC(4), 4-week Politis-Romano block bootstrap, Jarque-Bera / Breusch-Pagan / Durbin-Watson / Ljung-Box diagnostics, Student-t co-primary via `TLinearModel`, GARCH(1,1)-X via `arch_model`, and CPI/PPI decomposition co-primary — traces directly to the locked spec text. An edit to the spec that changed any of these choices would have flipped this hash and halted execution.


### §1.2 Panel fingerprint verification

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 handoff-artifact requirement; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], pre-commitment packaging of the downstream specification surface.

**Why used.** The Ankel-Peters protocol mandates that the handoff artifact record the exact data state the downstream step will consume — not a description of it, but a cryptographic fingerprint. NB1 §8b emitted `nb1_panel_fingerprint.json` carrying the sha256 of the weekly panel (947 rows × 18 columns) after Decision #12 listwise complete-case drop. NB2 §1.2 closes the loop: it re-loads the panel via the same `cleaning.load_cleaned_panel` entry point and re-computes the fingerprint. Any divergence — a DuckDB repopulation, a silent `cleaning.py` edit, a schema change, even a row-order shift that escaped the sort — flips the hash and halts NB2 before estimation.

**Relevance to our results.** All twelve NB1 Decisions (#1 sample window through #12 merge policy) are embedded in the `cleaning.LOCKED_DECISIONS` dataclass and reflected byte-for-byte in the weekly-panel sha256. Verifying the fingerprint here is the single check that guarantees the 947-observation Column-6 primary OLS in §3 runs on the pre-registered data. Without this check, a regenerated DuckDB with even one different row would silently propagate through every β̂, Newey-West SE, and bootstrap CI in §3-§7.

**Connection to simulator.** Layer 2's FHS innovation pool — the simulator's residual-draw surface — is conditioned on the exact OLS residuals Column 6 emits. Those residuals are a pure function of (a) the weekly panel and (b) the locked spec. §1.1 verified the spec; §1.2 verifies the panel. Together they pin the entire downstream fitted-parameter surface that the simulator consumes.


In [ ]:
# §1.2 Panel fingerprint verification (Decision #0 continues).
#
# Re-loads the cleaned weekly panel via cleaning.load_cleaned_panel and
# fingerprints it via panel_fingerprint.fingerprint on the "week_start"
# date column. Compares the recomputed sha256 to the NB1 handoff JSON's
# weekly_panel.sha256 field. Any drift halts.
import json

import duckdb

from scripts import cleaning
from scripts import panel_fingerprint

_handoff = json.loads(env.FINGERPRINT_PATH.read_text(encoding="utf-8"))
_nb1_weekly_sha = _handoff["weekly_panel"]["sha256"]

# Embedded lock (redundant with _handoff but explicit for code review).
_EMBEDDED_WEEKLY_SHA256 = (
    "769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f"
)

# Pre-flight: the handoff JSON must match our embedded lock. Catches
# the case where someone silently edits both nb1_panel_fingerprint.json
# and cleaning.py together — the embedded literal would still diverge.
assert _nb1_weekly_sha == _EMBEDDED_WEEKLY_SHA256, (
    f"Handoff JSON weekly sha256 {_nb1_weekly_sha} differs from the "
    f"embedded Task-16 lock {_EMBEDDED_WEEKLY_SHA256}. Halt."
)

_conn_nb2 = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _cleaned = cleaning.load_cleaned_panel(_conn_nb2)
finally:
    _conn_nb2.close()

_nb2_weekly_fp = panel_fingerprint.fingerprint(
    _cleaned.weekly, date_column="week_start"
)
_nb2_weekly_sha = _nb2_weekly_fp["sha256"]

assert _nb2_weekly_sha == _nb1_weekly_sha, (
    f"Panel fingerprint drift: NB2 recomputed sha256 {_nb2_weekly_sha} "
    f"does not match NB1 handoff {_nb1_weekly_sha} at "
    f"{env.FINGERPRINT_PATH}. Halt — estimator must not run on a "
    f"drifted panel."
)

# Bind the cleaned panel + weekly DataFrame into the notebook namespace
# so §2-§7 can consume them without re-opening DuckDB.
panel = _cleaned
weekly = _cleaned.weekly

print(f"Panel fingerprint verified: weekly sha256 = {_nb2_weekly_sha}")
print(f"  rows = {_nb2_weekly_fp['row_count']}, "
      f"cols = {_nb2_weekly_fp['column_count']}")
print(f"  date range: {_nb2_weekly_fp['date_min']} → "
      f"{_nb2_weekly_fp['date_max']}")
print(f"Decision hash = {_cleaned.decision_hash}")


**Interpretation — §1.2.** The weekly panel on which §3's Column-6 primary OLS will fit is byte-identical to the panel NB1 committed on 2026-04-18. All twelve Decisions (#1 sample window 2008-01-02→2026-03-01, #2 RV^{1/3}, #3 weekly frequency, #4 CPI surprise AR(1), #5 US CPI 12-month warmup, #6 BanRep event-study ΔIBR, #7 VIX weekly mean, #8 oil last-positive log-return, #9 intervention binary, #10 no collinearity adjustment, #11 levels no differencing, #12 listwise complete-case) survive the fingerprint check. The decision hash `6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c` is the single string that binds the estimation we are about to run to the pre-registered lock.

> **Gate Verdict:** populated after NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary here once §3-§7 estimation, §4 diagnostics, and §3.5 block-bootstrap sensitivity have been executed end-to-end.


## 2. Descriptive statistics — full-sample

### §2.1 Full-sample central moments of LHS and RHS regressors

**Reference.** Conrad, Schoelkopf and Tushteva (2025), "Long-term volatility shapes the stock market's sensitivity to news," *Journal of Applied Econometrics* [@conrad2025longterm], Table 1 descriptive-statistics convention; Ankel-Peters, Brodeur, Connolly and Schwandt (2024) [@ankelPeters2024protocol], §3 reporting discipline for replication-protocol pipelines.

**Why used.** Conrad-Schoelkopf-Tushteva's Table 1 is the genre template for what a pre-registered structural-econometrics paper's descriptive-stats block does and does not show. The convention: report mean, standard deviation, skewness, and kurtosis for the dependent variable and every right-hand-side regressor, computed over the full estimation sample, with no release-week conditioning, no sub-sample splits, and no test statistics. Those are §3-§7 concerns; §2's role is strictly the reader's first-pass sanity check on the data's central moments.

**Relevance to our results.** The §3 OLS ladder, §3.5 Politis-Romano bootstrap, §4 JB/BP/DW/LB diagnostics, §5 Student-t refit, §6 GARCH(1,1)-X, and §7 CPI/PPI decomposition all lean on the seven variables enumerated below. Reporting their full-sample central moments before any fit runs lets the reviewer verify — without yet having seen a β̂ — that the data's location, scale, skewness, and tail behavior are consistent with the literature (weekly RV^{1/3} moments from Andersen-Bollerslev-Diebold-Labys 2001, surprise-series skewness from ABDV 2003, VIX kurtosis from Conrad et al.) before inspecting any coefficient.

**Connection to simulator.** Layer 2's FHS innovation pool draws residuals from the Column-6 fit's standardized residual distribution; the shape of that distribution inherits the LHS's skewness and kurtosis almost mechanically. Surfacing those moments here — before the fit — is the simulator's sanity check that the residual pool it will eventually consume is not drawn from a pathological distribution an outlier hid.


In [ ]:
# §2.1 Full-sample descriptive statistics (mean / std / skew / kurtosis).
#
# Seven series in Column-6 order: LHS first, then the six RHS controls
# in the nested-ladder order (CPI surprise first — identifying
# regressor — then US CPI, BanRep rate surprise, VIX, oil return,
# intervention dummy). Full-sample only — no release-week filter, no
# sub-sample split, no test statistics. Those live in §3+.
import pandas as pd

_SERIES_ORDER = (
    "rv_cuberoot",
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "oil_return",
    "intervention_dummy",
)

# Extract just the seven columns of interest from the weekly panel.
# intervention_dummy is int16; coerce to float so skew/kurtosis apply.
_series_df = weekly[list(_SERIES_ORDER)].astype("float64")

_descriptive_stats = pd.DataFrame(
    {
        "mean": _series_df.mean(),
        "std": _series_df.std(ddof=1),
        "skew": _series_df.skew(),
        "kurtosis": _series_df.kurtosis(),
    }
).loc[list(_SERIES_ORDER)]

# Round for presentation but keep float dtype so downstream code can
# still use the table numerically if required.
_descriptive_stats.round(4)


**Interpretation — §2.1.** The descriptive-stats table above is the first of two pre-estimation sanity checks (the second — correlation + VIF — lives in §2.2 of a later task). The seven central-moment quartets are reported full-sample over 947 weekly observations spanning 2008-01-07 through 2026-02-23.

Expected patterns the reviewer should confirm before moving on to §3:

- `rv_cuberoot` is strictly positive with moderate skew and excess kurtosis consistent with ABDV 2001's characterization of realized-volatility cube-roots as approximately Gaussian but with detectable right-tail mass in crisis weeks.
- `cpi_surprise_ar1` — the identifying regressor — is centered close to zero by construction (AR(1) residual on IPC monthly changes), with skewness and kurtosis modest enough that the OLS ladder in §3 is not dominated by a handful of outliers.
- `us_cpi_surprise`, `banrep_rate_surprise`, `oil_return`, and `vix_avg` report the expected shapes for macro/commodity/risk controls: near-zero mean for surprises and returns, positive mean for VIX (levels), and the heavy-tailed VIX kurtosis that motivates the §3.5 block-bootstrap sensitivity check.
- `intervention_dummy` is bounded in {0, 1}; its mean reports the unconditional intervention probability over the full sample.

If any of these patterns look anomalous — e.g. a negative mean on `vix_avg`, kurtosis below 0 on `rv_cuberoot`, or |skew| above 5 on a surprise series — halt before §3 and re-inspect the DuckDB or `cleaning.py` rather than proceeding to estimation on a panel that disagrees with the literature.

No Decision fires in §2; all twelve locks carry over from NB1 and have already been verified in §1.2.
